# Tách CSV theo nhãn (clean / offensive / hate) ra Excel nhiều sheet

Đọc file CSV lớn (streaming, không load hết vào RAM) và ghi mỗi nhãn vào 1 hoặc
nhiều sheet Excel để kiểm lại thủ công.

- Nếu 1 nhãn có nhiều hơn **1,048,575 dòng** (giới hạn của Excel), sẽ tự động
  tách thêm sheet (`offensive`, `offensive_2`, ...).
- Chỉnh các biến ở ô **Cấu hình** bên dưới rồi chạy lần lượt các ô.

In [2]:
import csv
import os
import sys
import tempfile

from openpyxl import Workbook

MAX_ROWS_PER_SHEET = 1_048_575  # trừ 1 dòng cho header

## Cấu hình — chỉnh đường dẫn & tên cột tại đây

In [3]:
INPUT_CSV = "E:\PythonAI\PBL6\ToxicContentDetection\data\labeled\labeled_model_qwen.csv"
OUTPUT_XLSX = "E:\PythonAI\PBL6\ToxicContentDetection\data\labeled\labeled_model_qwen_split.xlsx"
LABEL_COL = "label"
ENCODING = "utf-8"
DELIMITER = ","

if not os.path.exists(INPUT_CSV):
    raise SystemExit(f"Không tìm thấy file: {INPUT_CSV}")

## Các hàm xử lý

In [4]:
def normalize_label(raw_label: str) -> str:
    """Chuẩn hoá nhãn để gom nhóm (bỏ khoảng trắng thừa, hạ chữ thường)."""
    return raw_label.strip().lower()


INVALID_SHEET_CHARS = set(['[', ']', ':', '*', '?', '/', chr(92)])  # chr(92) = backslash


def safe_sheet_name(label: str, part: int) -> str:
    """Excel giới hạn tên sheet <= 31 ký tự và không cho vài ký tự đặc biệt."""
    base = "".join(c for c in label if c not in INVALID_SHEET_CHARS)[:25] or "sheet"
    return base if part == 1 else f"{base}_{part}"[:31]

In [5]:
def split_by_label(input_csv, label_col, encoding, delimiter, tmp_dir):
    """
    Đọc file CSV gốc 1 lần duy nhất (streaming), ghi mỗi dòng vào 1 file CSV
    tạm riêng theo nhãn. Trả về dict: label -> (tmp_path, row_count).
    """
    label_files = {}   # label -> [file_handle, csv_writer, path, row_count]
    header = None

    with open(input_csv, "r", encoding=encoding, newline="") as f:
        reader = csv.reader(f, delimiter=delimiter)
        header = next(reader)
        try:
            label_idx = header.index(label_col)
        except ValueError:
            raise SystemExit(
                f"Không tìm thấy cột '{label_col}' trong header: {header}\n"
                f"-> Sửa lại biến LABEL_COL cho khớp"
            )

        total = 0
        for row in reader:
            total += 1
            if total % 500_000 == 0:
                print(f"  Đã đọc {total:,} dòng...")

            if len(row) <= label_idx:
                continue  # dòng lỗi format, bỏ qua

            label = normalize_label(row[label_idx])

            if label not in label_files:
                path = os.path.join(tmp_dir, f"{label}.csv")
                fh = open(path, "w", encoding="utf-8", newline="")
                writer = csv.writer(fh)
                writer.writerow(header)
                label_files[label] = [fh, writer, path, 0]

            entry = label_files[label]
            entry[1].writerow(row)
            entry[3] += 1

    for entry in label_files.values():
        entry[0].close()

    print(f"Tổng số dòng đã xử lý: {total:,}")
    return {label: (v[2], v[3]) for label, v in label_files.items()}, header

In [6]:
def write_label_to_sheets(wb, label, tmp_path, header, row_count):
    """Đọc file CSV tạm của 1 nhãn, ghi vào 1 hoặc nhiều sheet (write-only)."""
    with open(tmp_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader)  # bỏ header cũ, sẽ tự ghi lại cho từng sheet

        part = 1
        ws = wb.create_sheet(title=safe_sheet_name(label, part))
        ws.append(header)
        count_in_sheet = 0

        for row in reader:
            if count_in_sheet >= MAX_ROWS_PER_SHEET:
                part += 1
                ws = wb.create_sheet(title=safe_sheet_name(label, part))
                ws.append(header)
                count_in_sheet = 0
            ws.append(row)
            count_in_sheet += 1

    n_sheets = part
    print(f"  Nhãn '{label}': {row_count:,} dòng -> {n_sheets} sheet")
    return n_sheets

## Chạy xử lý

In [7]:
with tempfile.TemporaryDirectory() as tmp_dir:
    print("Bước 1/2: Đang tách dữ liệu theo nhãn (streaming)...")
    label_files, header = split_by_label(INPUT_CSV, LABEL_COL, ENCODING, DELIMITER, tmp_dir)

    if not label_files:
        raise SystemExit("Không đọc được dòng dữ liệu nào.")

    print(f"Tìm thấy {len(label_files)} nhãn: {list(label_files.keys())}")

    print("Bước 2/2: Đang ghi ra file Excel (mỗi nhãn 1+ sheet)...")
    wb = Workbook(write_only=True)
    total_sheets = 0

    # Ưu tiên thứ tự quen thuộc nếu có, còn lại xếp theo alphabet
    preferred_order = ["clean", "offensive", "hate"]
    rest_labels = sorted(l for l in label_files if l not in preferred_order)
    ordered_labels = [l for l in preferred_order if l in label_files] + rest_labels

    for label in ordered_labels:
        tmp_path, row_count = label_files[label]
        total_sheets += write_label_to_sheets(wb, label, tmp_path, header, row_count)

    wb.save(OUTPUT_XLSX)

print(f"\nHoàn tất! Đã tạo {total_sheets} sheet trong: {OUTPUT_XLSX}")

Bước 1/2: Đang tách dữ liệu theo nhãn (streaming)...
  Đã đọc 500,000 dòng...
  Đã đọc 1,000,000 dòng...
  Đã đọc 1,500,000 dòng...
  Đã đọc 2,000,000 dòng...
  Đã đọc 2,500,000 dòng...
  Đã đọc 3,000,000 dòng...
  Đã đọc 3,500,000 dòng...
  Đã đọc 4,000,000 dòng...
  Đã đọc 4,500,000 dòng...
  Đã đọc 5,000,000 dòng...
  Đã đọc 5,500,000 dòng...
Tổng số dòng đã xử lý: 5,678,795
Tìm thấy 3 nhãn: ['clean', 'offensive', 'hate']
Bước 2/2: Đang ghi ra file Excel (mỗi nhãn 1+ sheet)...
  Nhãn 'clean': 4,166,251 dòng -> 4 sheet
  Nhãn 'offensive': 1,259,254 dòng -> 2 sheet
  Nhãn 'hate': 253,290 dòng -> 1 sheet

Hoàn tất! Đã tạo 7 sheet trong: E:\PythonAI\PBL6\ToxicContentDetection\data\labeled\labeled_model_qwen_split.xlsx
